# CodeLlama-v9 – Smart Contract Vulnerability Detector

## QLoRA 4-bit Fine-Tuning trên `final-dataset-train.json` (26,652 mẫu)

Huấn luyện mô hình **CodeLlama-7B** chuyên biệt phân loại nhị phân **`Safe`** vs **`Vulnerable`** cho các hàm Solidity smart contract.

### Đặc điểm Dataset `final-dataset-train.json`:
- **Tổng 26,652 hàm** function-level từ 3 nguồn thực tế: `kaggle_ds2` (20,896), `iaudit_trustllm` (3,111), `dappscan` (2,645).
- **Phân chia chuẩn** đã sẵn trong dataset:
  - **Train** (21,322 mẫu - 80%): Safe 10,980 / Vulnerable 10,342 — gần cân bằng nhất.
  - **Val** (2,665 mẫu - 10%): Safe 1,390 / Vulnerable 1,275 — Early Stopping & Threshold Tuning.
  - **Test** (2,665 mẫu - 10%): Safe 1,398 / Vulnerable 1,267 — Benchmark kiểm thử độc lập.
- **Code ngắn gọn**: Median 186 chars (5 LOC), P95 = 781 chars, P99 = 1,703 chars. ~96.8% mẫu < 256 tokens.

### Chiến lược huấn luyện ưu tiên chất lượng:
- **LoRA rank 32** + alpha 64 + target all 7 linear layers → nhiều tham số trainable hơn → khả năng biểu diễn tốt hơn.
- **MAX_LEN = 1024** bao phủ 99.93% mẫu, không mất thông tin.
- **Gradient Accumulation 32** → effective batch size 32 với BS=1 trên T4 → ổn định gradient trên dataset lớn.
- **Epochs 6 + Early Stopping patience 3** — đủ lâu để hội tụ trên 21K mẫu.
- **Cosine scheduler + label smoothing 0.02** → tránh overfit, cải thiện calibration.
- **Class-Balanced Sampler** cân bằng Safe/Vulnerable trong mỗi epoch.
- **Threshold Tuning** tự động trên Val set trước khi đánh giá Test.

### Hướng dẫn chạy trên Kaggle:
1. **Add Data** → Upload tệp `final-dataset-train.json` (hoặc `.csv` / `.jsonl`).
2. **Bật GPU** (T4 x1 hoặc T4 x2, Accelerator: GPU T4).
3. Chạy cell 0 cài đặt, **Restart Kernel**, rồi chọn **Run All**.
4. Nếu CodeLlama bị gate trên Hugging Face: Add Input → Models → `CodeLlama-7B` hoặc khai báo `HF_TOKEN` trong Kaggle Secrets.

## 0. Cài đặt package & môi trường (Tương thích Python 3.12)

In [ ]:
# Cài đặt phiên bản mới nhất tương thích hoàn toàn với Python 3.12 trên Kaggle GPU T4
!pip install -q -U "bitsandbytes>=0.46.1" "transformers>=4.48.0" "accelerate>=1.0.1" "peft>=0.13.2" "tokenizers>=0.20.3" scikit-learn sentencepiece
!pip uninstall -y torchao
print("✅ Cài đặt hoàn tất → Chọn Run → Restart Kernel → Run All")

## 1. Cấu hình Hyperparameters

In [ ]:
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # Tránh lỗi DataParallel khi dùng 2xT4

import json, glob, random, collections, re, csv, inspect, math
from pathlib import Path
import numpy as np, torch
import transformers
from torch.utils.data import WeightedRandomSampler
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding,
)
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ─── Model base CodeLlama 7B ───
HF_MODEL = "codellama/CodeLlama-7b-hf"

# ─── Context & Batch Size (GPU T4 16GB VRAM) ───
MAX_LEN = 1024      # Bao phủ 99.93% mẫu trong dataset
TRAIN_BS = 1         # Batch size 1 trên GPU T4 để tối ưu VRAM
EVAL_BS = 2          # Batch size 2 cho Validation
GRAD_ACCUM = 32      # Effective batch size = 1 × 32 = 32 (ổn định trên 21K mẫu)

# ─── Training Schedule (ưu tiên chất lượng) ───
EPOCHS = 10          # Đủ epoch để hội tụ trên dataset 21K mẫu
LR = 1e-4            # Learning rate thấp hơn → ổn định, tránh overfit trên dataset lớn
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06  # 6% warmup → khoảng 2 epoch khởi động mượt
PATIENCE = 3         # Early Stopping sau 3 epoch không cải thiện
LABEL_SMOOTHING = 0.02
USE_GRADIENT_CHECKPOINTING = True

# ─── LoRA Config (rank cao hơn → biểu diễn tốt hơn) ───
LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05
LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# ─── Class Balance ───
USE_CLASS_BALANCED_SAMPLER = True
CLASS_BALANCE_POWER = 0.7

# ─── Threshold Tuning ───
THRESHOLD_OBJECTIVE = "f1"
THRESHOLD_GRID = np.round(np.arange(0.05, 0.951, 0.005), 3)

LABEL2ID = {"Safe": 0, "Vulnerable": 1}
ID2LABEL = {0: "Safe", 1: "Vulnerable"}
OUTPUT_DIR = "./codellama-v9-final-dataset-detector"
SAVE_DIR = OUTPUT_DIR + "-final"

print(
    "transformers:", transformers.__version__,
    "| device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    "| base:", HF_MODEL,
    "| mode: QLoRA 4-bit (final-dataset-train 26,652 samples)",
    "| max_len:", MAX_LEN,
    "| effective_bs:", TRAIN_BS * GRAD_ACCUM,
)

## 2. Tìm Model & Nạp Dataset `final-dataset-train.json` (26,652 mẫu)

In [ ]:
def find_model():
    """Tìm CodeLlama model: ưu tiên file local trên Kaggle, fallback về HuggingFace."""
    candidates = []
    for cfg in glob.glob("/kaggle/input/**/config.json", recursive=True):
        model_dir = os.path.dirname(cfg)
        try:
            model_type = json.load(open(cfg, encoding="utf-8")).get("model_type", "")
        except Exception:
            continue
        weights = (
            glob.glob(model_dir + "/*.safetensors")
            or glob.glob(model_dir + "/pytorch_model*.bin")
            or glob.glob(model_dir + "/*.index.json")
        )
        if model_type == "llama" and weights:
            candidates.append(model_dir)
    if candidates:
        candidates.sort(key=len)
        print("MODEL local:", candidates[0])
        return candidates[0]
    print("MODEL Hugging Face:", HF_MODEL)
    return HF_MODEL


def find_data():
    """
    Tìm dataset: hỗ trợ .json, .jsonl, .csv.
    Ưu tiên tìm final-dataset-train trước, fallback sang các tên cũ.
    """
    patterns = [
        # ── final-dataset-train (ưu tiên) ──
        "/kaggle/input/**/final-dataset-train.json",
        "/kaggle/input/**/final-dataset-train.jsonl",
        "/kaggle/input/**/final-dataset-train.csv",
        "final-dataset-train.json",
        "final-dataset-train.jsonl",
        "final-dataset-train.csv",
        # ── fallback local paths ──
        "final_dataset_train/final-dataset-train.json",
        "../final_dataset_train/final-dataset-train.json",
    ]
    for pattern in patterns:
        hits = sorted(glob.glob(pattern, recursive=True))
        if hits:
            return hits[0]
    raise FileNotFoundError(
        "Chưa thấy dataset! Vui lòng Add Data: upload final-dataset-train.json (hoặc .jsonl/.csv)"
    )


MODEL_PATH = find_model()
DATA_PATH = find_data()
print("DATASET:", DATA_PATH)

# ─── Nạp dữ liệu (hỗ trợ 3 định dạng) ───
if DATA_PATH.endswith(".csv"):
    with open(DATA_PATH, encoding="utf-8") as f:
        raw_rows = list(csv.DictReader(f))
    # CSV lưu label dạng string "0"/"1", cần chuyển
    for r in raw_rows:
        if "label" in r and isinstance(r["label"], str) and r["label"].isdigit():
            r["label"] = int(r["label"])
elif DATA_PATH.endswith(".jsonl"):
    raw_rows = [json.loads(line) for line in open(DATA_PATH, encoding="utf-8") if line.strip()]
else:
    with open(DATA_PATH, encoding="utf-8") as f:
        raw_rows = json.load(f)

# ─── Lọc mẫu hợp lệ: có code + label hợp lệ ───
rows = []
for r in raw_rows:
    if not r.get("code"):
        continue
    # Chuẩn hóa label: hỗ trợ cả int (0/1) và string ("Safe"/"Vulnerable")
    label = r.get("label")
    label_text = r.get("label_text", "")
    if isinstance(label, int) and label in (0, 1):
        r["label_text"] = ID2LABEL[label]
    elif isinstance(label, str) and label in LABEL2ID:
        r["label_text"] = label
        r["label"] = LABEL2ID[label]
    elif label_text in LABEL2ID:
        r["label"] = LABEL2ID[label_text]
    else:
        continue
    rows.append(r)

print(f"Tổng số mẫu hợp lệ: {len(rows)} / {len(raw_rows)} nạp")

# ─── Phân chia theo trường split có sẵn trong dataset ───
splits = {"train": [], "val": [], "test": []}
for r in rows:
    split_name = r.get("split", "train")
    splits.get(split_name, splits["train"]).append(r)

sep = "=" * 70
print(f"\n{sep}")
print("PHÂN BỔ CÁC TẬP SPLIT (final-dataset-train.json)")
print(sep)
role_map = {"train": "Huấn luyện", "val": "Validation + Threshold Tuning", "test": "Benchmark Kiểm Thử"}
for split_name, items in splits.items():
    n_vuln = sum(1 for x in items if x["label"] == 1)
    n_safe = len(items) - n_vuln
    by_source = dict(collections.Counter(x.get("source", "unknown") for x in items))
    role_desc = role_map.get(split_name, "")
    print(f"  {split_name:5s}: {len(items):6d} mẫu (Safe {n_safe:5d} / Vuln {n_vuln:5d}) — {role_desc}")
    print(f"         Nguồn: {by_source}")
print(sep)

## 3. Tokenize & Class-Balanced Sampler

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"


def build_model_input(row):
    """Xây dựng input text cho model: chỉ dùng raw code, không thêm prefix."""
    return row["code"].strip()


def build_class_label_weights(items):
    """Tính sample weights cho Class-Balanced Sampler (cân bằng Safe/Vulnerable)."""
    labels = [ID2LABEL[r["label"]] for r in items]
    counts = collections.Counter(labels)
    n_classes = max(1, len(counts))
    base = []
    for l in labels:
        w = (len(items) / (n_classes * counts[l])) ** CLASS_BALANCE_POWER
        base.append(w)
    weights = torch.as_tensor(base, dtype=torch.double)
    weights = weights / weights.mean()
    print("\nTrain class sampler distribution:")
    for l in sorted(counts):
        class_weights = [weights[i].item() for i, ll in enumerate(labels) if ll == l]
        print(f"  {l:12s}: n={counts[l]:5d} mean_weight={np.mean(class_weights):.4f}")
    return weights


class VulnDataset(torch.utils.data.Dataset):
    """Dataset wrapper cho sequence classification."""
    def __init__(self, items, sample_weights=None):
        self.items = items
        self.texts = [build_model_input(r) for r in items]
        self.enc = tokenizer(
            self.texts,
            truncation=True,
            max_length=MAX_LEN,
            add_special_tokens=True,
        )
        self.labels = [r["label"] for r in items]
        self.sample_weights = sample_weights

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: self.enc[k][idx] for k in self.enc}
        item["labels"] = self.labels[idx]
        return item


print("Đang tokenize 3 tập (train/val/test)...")
train_weights = build_class_label_weights(splits["train"]) if USE_CLASS_BALANCED_SAMPLER else None
train_ds = VulnDataset(splits["train"], sample_weights=train_weights)
val_ds = VulnDataset(splits["val"])
test_ds = VulnDataset(splits["test"])
collator = DataCollatorWithPadding(tokenizer)

# Thống kê token bị cắt
ntr = sum(1 for text in train_ds.texts if len(tokenizer(text, add_special_tokens=True)["input_ids"]) > MAX_LEN)
print(f"\nTrain bị cắt >{MAX_LEN} token: {ntr}/{len(train_ds)} ({100*ntr/len(train_ds):.2f}%)")

# Thống kê token length
tok_lens = [len(tokenizer(t, add_special_tokens=True)["input_ids"]) for t in train_ds.texts[:2000]]
print(f"Token length (2000 mẫu đầu): mean={np.mean(tok_lens):.0f}, median={np.median(tok_lens):.0f}, "
      f"p95={np.percentile(tok_lens,95):.0f}, max={max(tok_lens)}")
print(f"\nVí dụ input mẫu:\n{train_ds.texts[0][:500]}")

## 4. Nạp CodeLlama 7B (4-bit NF4) + QLoRA

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH,
    num_labels=2,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    quantization_config=bnb_config,
    device_map={"": 0},
    dtype=torch.float16,
)

model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=USE_GRADIENT_CHECKPOINTING)
lora = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    target_modules=LORA_TARGETS,
    modules_to_save=["score"],  # Classification head cần trainable đầy đủ
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

model.is_parallelizable = True
model.model_parallel = True

## 5. Huấn luyện mô hình (Fine-Tuning)

In [ ]:
def logits_from_predictions(predictions):
    if isinstance(predictions, tuple):
        predictions = predictions[0]
    return np.asarray(predictions)


def probs_from_logits(logits):
    logits = np.asarray(logits)
    logits = logits - logits.max(axis=-1, keepdims=True)
    exp = np.exp(logits)
    return exp[:, 1] / exp.sum(axis=-1)


def paper_metrics(labels, pred):
    """Metrics chuẩn theo bài báo: macro-averaged precision, recall, F1."""
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, pred, average="macro", zero_division=0
    )
    return {
        "accuracy": accuracy_score(labels, pred),
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }


def compute_metrics(eval_pred):
    logits = logits_from_predictions(eval_pred.predictions)
    labels = eval_pred.label_ids
    pred = np.argmax(logits, axis=-1)
    m = paper_metrics(labels, pred)
    return {
        "accuracy": m["accuracy"],
        "precision": m["precision"],
        "recall": m["recall"],
        "f1_macro": m["f1"],
        "f1_vuln": f1_score(labels, pred, pos_label=1, zero_division=0),
    }


class ClassBalancedTrainer(Trainer):
    """Custom Trainer với WeightedRandomSampler để cân bằng class."""
    def _default_train_sampler(self, train_dataset=None):
        method = super()._get_train_sampler
        if len(inspect.signature(method).parameters) == 0:
            return method()
        return method(train_dataset)

    def _get_train_sampler(self, train_dataset=None):
        dataset = train_dataset if train_dataset is not None else self.train_dataset
        if not USE_CLASS_BALANCED_SAMPLER:
            return self._default_train_sampler(train_dataset)
        if dataset is None or getattr(dataset, "sample_weights", None) is None:
            return self._default_train_sampler(train_dataset)
        generator = torch.Generator()
        generator.manual_seed(SEED)
        return WeightedRandomSampler(
            weights=dataset.sample_weights,
            num_samples=len(dataset.sample_weights),
            replacement=True,
            generator=generator,
        )


# ─── Xây dựng TrainingArguments tương thích linh hoạt các phiên bản transformers ───
training_args_kwargs = {
    "output_dir": OUTPUT_DIR,
    "num_train_epochs": EPOCHS,
    "per_device_train_batch_size": TRAIN_BS,
    "per_device_eval_batch_size": EVAL_BS,
    "gradient_accumulation_steps": GRAD_ACCUM,
    "learning_rate": LR,
    "weight_decay": WEIGHT_DECAY,
    "warmup_ratio": WARMUP_RATIO,
    "lr_scheduler_type": "cosine",
    "fp16": True,
    "optim": "paged_adamw_8bit",
    "gradient_checkpointing": USE_GRADIENT_CHECKPOINTING,
    "label_smoothing_factor": LABEL_SMOOTHING,
    "max_grad_norm": 0.3,
    "eval_strategy": "epoch",
    "save_strategy": "epoch",
    "load_best_model_at_end": True,
    "metric_for_best_model": "f1_macro",
    "greater_is_better": True,
    "logging_steps": 50,       # Log mỗi 50 step (phù hợp 21K mẫu)
    "save_total_limit": 2,
    "report_to": "none",
    "seed": SEED,
    "remove_unused_columns": False,
    "dataloader_pin_memory": False,
}


def build_training_args(kwargs):
    """Xây dựng TrainingArguments, tự động xử lý tương thích phiên bản."""
    kwargs = dict(kwargs)
    signature = inspect.signature(TrainingArguments.__init__)
    parameters = signature.parameters
    accepts_kwargs = any(p.kind == inspect.Parameter.VAR_KEYWORD for p in parameters.values())
    supported = set(parameters) - {"self"}

    if not accepts_kwargs:
        if "eval_strategy" in kwargs and "eval_strategy" not in supported and "evaluation_strategy" in supported:
            kwargs["evaluation_strategy"] = kwargs.pop("eval_strategy")
        elif "evaluation_strategy" in kwargs and "evaluation_strategy" not in supported and "eval_strategy" in supported:
            kwargs["eval_strategy"] = kwargs.pop("evaluation_strategy")

        if "warmup_ratio" in kwargs and "warmup_ratio" not in supported and "warmup_steps" in supported:
            steps_per_epoch = max(1, math.ceil(len(train_ds) / max(1, TRAIN_BS) / max(1, GRAD_ACCUM)))
            kwargs["warmup_steps"] = int(steps_per_epoch * EPOCHS * WARMUP_RATIO)
            kwargs.pop("warmup_ratio")

        dropped = sorted(k for k in kwargs if k not in supported)
        for key in dropped:
            kwargs.pop(key)
        if dropped:
            print("Dropped unsupported TrainingArguments:", ", ".join(dropped))

    return TrainingArguments(**kwargs)


args = build_training_args(training_args_kwargs)

trainer_kwargs = {
    "model": model,
    "args": args,
    "train_dataset": train_ds,
    "eval_dataset": val_ds,
    "data_collator": collator,
    "compute_metrics": compute_metrics,
    "callbacks": [EarlyStoppingCallback(early_stopping_patience=PATIENCE)],
}
try:
    trainer = ClassBalancedTrainer(processing_class=tokenizer, **trainer_kwargs)
except TypeError:
    trainer = ClassBalancedTrainer(tokenizer=tokenizer, **trainer_kwargs)

trainer.args._n_gpu = 1

steps_per_epoch = max(1, math.ceil(len(train_ds) / max(1, TRAIN_BS) / max(1, GRAD_ACCUM)))
total_steps = steps_per_epoch * EPOCHS
sep = "=" * 60
print(f"\n{sep}")
print("BẮT ĐẦU HUẤN LUYỆN")
print(f"  Train mẫu:    {len(train_ds):,}")
print(f"  Val mẫu:      {len(val_ds):,}")
print(f"  Epochs:       {EPOCHS}")
print(f"  Steps/epoch:  ~{steps_per_epoch}")
print(f"  Total steps:  ~{total_steps}")
print(f"  Effective BS: {TRAIN_BS * GRAD_ACCUM}")
print(f"  LoRA rank:    {LORA_R}")
print(sep + "\n")

trainer.train()

print("\nEpoch | val_acc | val_prec | val_recall | val_f1 | val_f1_vuln")
for h in trainer.state.log_history:
    if "eval_f1_macro" in h:
        ep = round(h["epoch"])
        print(
            f"  {ep:3d} | {h['eval_accuracy']:.4f} | {h['eval_precision']:.4f} | "
            f"{h['eval_recall']:.4f} | {h['eval_f1_macro']:.4f} | {h['eval_f1_vuln']:.4f}"
        )

## 6. Tìm Threshold tối ưu trên Validation (2,665 mẫu)

In [ ]:
def metrics_at_threshold(labels, probs, threshold):
    pred = (probs >= threshold).astype(int)
    return paper_metrics(labels, pred)


def tune_threshold(labels, probs, thresholds=THRESHOLD_GRID, objective=THRESHOLD_OBJECTIVE):
    rows_th = []
    for th in thresholds:
        m = metrics_at_threshold(labels, probs, th)
        rows_th.append({"threshold": float(th), **m})
    best = max(rows_th, key=lambda r: (r[objective], r["f1"], r["recall"], r["precision"]))
    return best, rows_th


val_out = trainer.predict(val_ds)
val_logits = logits_from_predictions(val_out.predictions)
val_probs = probs_from_logits(val_logits)
val_labels = val_out.label_ids
BEST_THRESHOLD_INFO, threshold_rows = tune_threshold(val_labels, val_probs)
BEST_THRESHOLD = BEST_THRESHOLD_INFO["threshold"]

print(f"Best threshold by validation {THRESHOLD_OBJECTIVE}: {BEST_THRESHOLD:.3f}")
print(
    f">> VAL: Accuracy={BEST_THRESHOLD_INFO['accuracy']:.4f}  "
    f"Precision={BEST_THRESHOLD_INFO['precision']:.4f}  "
    f"Recall={BEST_THRESHOLD_INFO['recall']:.4f}  F1={BEST_THRESHOLD_INFO['f1']:.4f}"
)

print("\nTop 10 thresholds trên Validation:")
header = f"{'thr':>6s} {'acc':>7s} {'prec':>7s} {'recall':>7s} {'f1':>7s}"
print(header)
for r in sorted(threshold_rows, key=lambda x: x[THRESHOLD_OBJECTIVE], reverse=True)[:10]:
    print(f"{r['threshold']:6.3f} {r['accuracy']:7.4f} {r['precision']:7.4f} {r['recall']:7.4f} {r['f1']:7.4f}")

## 7. Đánh giá kiểm thử trên tập TEST (2,665 mẫu)

In [ ]:
pred_out = trainer.predict(test_ds)
test_logits = logits_from_predictions(pred_out.predictions)
test_probs = probs_from_logits(test_logits)
yt = pred_out.label_ids
yp = (test_probs >= BEST_THRESHOLD).astype(int)

sep = "=" * 70
print(sep)
print(f"TEST (n={len(yt)}) — CodeLlama-v9 Final Dataset QLoRA — threshold={BEST_THRESHOLD:.3f}")
print(sep)
print(classification_report(yt, yp, target_names=["Safe", "Vulnerable"], digits=4, zero_division=0))
print("Confusion [[TN FP][FN TP]]:\n", confusion_matrix(yt, yp))
m = paper_metrics(yt, yp)
print(f"\n>> Accuracy={m['accuracy']:.4f}  Precision={m['precision']:.4f}  Recall={m['recall']:.4f}  F1={m['f1']:.4f}")

# Phân bố nhãn trên test set
safe_total = sum(yt == 0)
vuln_total = sum(yt == 1)
print(f"\nTest ground truth: Safe={safe_total} ({safe_total/len(yt):.1%}), Vulnerable={vuln_total} ({vuln_total/len(yt):.1%})")

# Đánh giá theo từng nguồn dữ liệu (nếu có trường source)
test_items = splits["test"]
sources_in_test = set(r.get("source", "unknown") for r in test_items)
if len(sources_in_test) > 1:
    print("\n--- Kết quả theo từng nguồn dữ liệu ---")
    for src in sorted(sources_in_test):
        idxs = [i for i, r in enumerate(test_items) if r.get("source", "unknown") == src]
        if len(idxs) < 10:
            continue
        src_yt = yt[idxs]
        src_yp = yp[idxs]
        sm = paper_metrics(src_yt, src_yp)
        print(f"  {src:20s}: n={len(idxs):5d}  Acc={sm['accuracy']:.4f}  F1={sm['f1']:.4f}")

## 8. Lưu Adapter QLoRA và Threshold Config

In [ ]:
os.makedirs(SAVE_DIR, exist_ok=True)
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

threshold_config = {
    "base_model": MODEL_PATH,
    "hf_model": HF_MODEL,
    "best_threshold": BEST_THRESHOLD,
    "threshold_objective": THRESHOLD_OBJECTIVE,
    "validation_metrics": BEST_THRESHOLD_INFO,
    "max_len": MAX_LEN,
    "label2id": LABEL2ID,
    "id2label": ID2LABEL,
    "train_mode": "CodeLlama v9 sequence classification QLoRA 4-bit (final-dataset-train 26,652 samples)",
    "dataset": "final-dataset-train.json",
    "dataset_splits": {
        "train": len(splits["train"]),
        "val": len(splits["val"]),
        "test": len(splits["test"]),
    },
    "dataset_sources": dict(collections.Counter(r.get("source", "unknown") for r in rows)),
    "dataset_total_samples": len(rows),
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "effective_batch_size": TRAIN_BS * GRAD_ACCUM,
    "epochs": EPOCHS,
    "learning_rate": LR,
    "label_smoothing": LABEL_SMOOTHING,
    "class_balanced_sampler": USE_CLASS_BALANCED_SAMPLER,
    "class_balance_power": CLASS_BALANCE_POWER,
    "test_metrics": m,
}
with open(Path(SAVE_DIR) / "threshold_config.json", "w", encoding="utf-8") as f:
    json.dump(threshold_config, f, ensure_ascii=False, indent=2)

print("✅ Đã lưu model & config tại:", SAVE_DIR)
print("Files:", os.listdir(SAVE_DIR))

## 9. Smoke Test Inference

In [ ]:
model.eval()

@torch.no_grad()
def predict(code_text, threshold=None):
    if threshold is None:
        threshold = BEST_THRESHOLD
    enc = tokenizer(
        code_text.strip(),
        truncation=True,
        max_length=MAX_LEN,
        add_special_tokens=True,
        return_tensors="pt",
    ).to(next(model.parameters()).device)
    prob_vuln = torch.softmax(model(**enc).logits, dim=-1)[0, 1].item()
    return ("Vulnerable" if prob_vuln >= threshold else "Safe"), prob_vuln


SMOKE_TESTS = [
    {
        "name": "SWC-107 Reentrancy",
        "expected": "Vulnerable",
        "code": """function withdraw(uint amount) public {
    require(balances[msg.sender] >= amount);
    (bool ok,) = msg.sender.call{value: amount}("");
    require(ok);
    balances[msg.sender] -= amount;
}""",
    },
    {
        "name": "SWC-104 Unchecked Call Return",
        "expected": "Vulnerable",
        "code": """function pay(address payable to, uint256 amount) public {
    to.call{value: amount}("");
}""",
    },
    {
        "name": "Access Control Missing",
        "expected": "Vulnerable",
        "code": """function setOwner(address newOwner) public {
    owner = newOwner;
}""",
    },
    {
        "name": "Safe Guarded Withdraw",
        "expected": "Safe",
        "code": """function withdraw(uint amount) public nonReentrant {
    require(balances[msg.sender] >= amount);
    balances[msg.sender] -= amount;
    (bool ok,) = msg.sender.call{value: amount}("");
    require(ok);
}""",
    },
    {
        "name": "Safe AccessControl onlyOwner",
        "expected": "Safe",
        "code": """function setLimit(uint256 _limit) external onlyOwner {
    require(_limit > 0, "invalid");
    limit = _limit;
}""",
    },
    {
        "name": "tx.origin Auth (Vulnerable)",
        "expected": "Vulnerable",
        "code": """function emergencyWithdraw() external {
    require(tx.origin == owner, "not owner");
    payable(msg.sender).transfer(address(this).balance);
}""",
    },
]

print(f"\nSmoke test với threshold = {BEST_THRESHOLD:.3f}")
header = f"{'Case':35s} {'Expect':12s} {'Pred':12s} {'P(vuln)':>8s}"
print(header)
all_pass = True
for t in SMOKE_TESTS:
    pred_label, p = predict(t["code"])
    status = "✅" if pred_label == t["expected"] else "❌"
    if pred_label != t["expected"]:
        all_pass = False
    print(f"{t['name']:35s} {t['expected']:12s} {pred_label:12s} {p:8.3f} {status}")

sep = "=" * 60
print(f"\n{sep}")
if all_pass:
    print("✅ TẤT CẢ SMOKE TESTS ĐỀU PASS!")
else:
    print("⚠️  Một số smoke test chưa pass. Kiểm tra lại threshold hoặc tiếp tục train thêm epoch.")
print(sep)